In [ ]:
import geopandas as gpd
import numpy as np
from shapely import LineString, MultiLineString
from shapely.ops import linemerge

from geo_data import data_handler, helpers

DATA_PATH = helpers.get_top_directory() / "data"

# Rivers data creation
I had a lot of trouble with creating the data for the rivers.

In [ ]:
deck_name = "Allgemeinwissen::01 🟦 Geografie 🌍::01.04 Flüsse Deutschlands 🏞️🇩🇪"
anki_rivers_df = data_handler.anki_to_df(deck_name)
anki_rivers_df = anki_rivers_df[anki_rivers_df["NoteType"] == "Adrian::Flüsse"]

# Some rivers have different names at OSM (mainly language reasons)
anki_rivers_df["osm_name"] = anki_rivers_df["Flussname"].replace(
    {
        "Donau": "Danube",
        "Elde": "Elde-Müritz-Wasserstraße",
        "Eger": "Ohře",
        "Mosel": "La Moselle",
        "Oder": "Odra",
        "Saar": "La Sarre",
    }
)
anki_rivers_df = anki_rivers_df.sort_values(by="osm_name")
anki_rivers_df = anki_rivers_df.reset_index(drop=True)

regex = [f"^{f}$" for f in anki_rivers_df["osm_name"]]
regex = "|".join(regex)

overpass_query = f"""[out:json][timeout:300];
relation["waterway"="river"]["name"~"{regex}"];
out geom;"""
print(overpass_query)

I used this query to gather the data from OSM using https://overpass-turbo.eu/. Here, I ran the query, waited, and then did *Export -> GeoJSON -> Download* to obtain a GeoJSON-file.

When I did this for the first time, I loaded the file using geopandas and compared it to the rivers that I want:
```python
rivers = gpd.read_file(DATA_PATH / "export.geojson")
rivers = rivers[rivers["type"] == "waterway"]  # clean
set(df["Flussname"]) - set(rivers["name"])
```
Output: `{'Donau', 'Elde', 'Mosel', 'Oder', 'Saar'}`

-> Some rivers couldn't be found because of their names. Therefore, I manually searched for those rivers on OSM (either the [map](https://www.openstreetmap.org/) or a [list of water stuff](https://wiki.openstreetmap.org/wiki/Germany/Gew%C3%A4sser/Fl%C3%BCsse_und_Kan%C3%A4le_D-F)). Additionally, the "Eger"-River in Germany is not the one that I want, but I want the one in the Czech Republic. This is how I manually created the dict for the replacements. After that, I created the query again and came up with an almost ready file. However, some rivers had e.g. the same names. Therefore, I started to take a look at the remaining stuff.

I think the most simple thing is to just find out the numeric OSM-Identifiers of the rivers that I do not want. Before, I removed the entries where the type wasn't "waterway" (the source points). Then I grouped by the name and noted the IDs I didn't need when there were duplicates:

In [ ]:
rivers = gpd.read_file(
    DATA_PATH / "raw_data" / "german_rivers_export_2025-11-13.geojson"
)
rivers = rivers[(rivers["type"] == "waterway")]

for name, df in rivers.groupby("name"):
    if len(df) > 1:
        print(name)
        display(df)

The ones I do not want are:
- 15074211
- 15347024
- 5213209
- 6799134
- 7298530

I removed these, cleaned up the result and confirmed that we have every river that we want.

In [ ]:
unwanted_ids = [15074211, 15347024, 5213209, 6799134, 7298530]
unwanted_ids_str = [f"relation/{i}" for i in unwanted_ids]
clean_rivers = rivers[~rivers.id.isin(unwanted_ids_str)]
clean_rivers = clean_rivers.sort_values(by="name")
clean_rivers = clean_rivers.reset_index(drop=True)

assert all(clean_rivers.name == anki_rivers_df.osm_name)
# -> they are the same!
clean_rivers["osm_name"] = clean_rivers.name
clean_rivers["name"] = anki_rivers_df.Flussname

I still had a problem with the geometries of this river. Mainly, they were MultiLineStrings; my dream would be to have only single line strings, one for each river. This is sometimes not possible (e.g. circles or other weird stuff), but often, the problem are tiny side arms that can easily be deleted or just the lines almost touching but not being connected (which they should be in my opinion). Therefore, I tried to clean the geometries as far as possible, to reduce the overhead and make everything easier for later.

In [ ]:
def merge_lines(
    geom: MultiLineString, threshold: int = 5_000, buffer_width: int = 1_000
) -> LineString | MultiLineString:
    """Merge the geometry's lines into as few lines as possible.

    The lines' endpoints are connected to the closest endpoints of the other lines, if
    their distance is lower than a specific threshold. If the are connected and
    "snapped", they are merged into a single line. Additionally, all lines that are
    positioned behind other lines (or a buffer around the other lines) are removed.

    Args:
        geom: The multilinestring where the lines should be merged.
        threshold: The distance threshold for which line endpoints are "snapped". The
            unit should be the same as the given geometry (probably a projection to
            meters, e.g. through Web Mercator `gdf.to_crs("EPSG:3857")` is recommended).
            Default is 5_000.
        buffer_width: Width of the buffer around the lines where other "contained" lines
            are removed. The unit should be the same as the given geometry (probably a
            projection to meters, e.g. through Web Mercator `gdf.to_crs("EPSG:3857")` is
             recommended). Default is 1_000.

    Returns:
        The resulting geometry after the merge. This is either a MultiLineString if not
        all parts could be merged or a single LineString if only one line results.
    """
    lines = list(geom.geoms)
    for i, line in enumerate(lines):
        end_points = np.array([np.array(line.coords)[[0, -1]] for line in lines])

        other_end_points = np.delete(end_points, i, axis=0).reshape((-1, 2))
        this_end_points = end_points[i]

        diff = other_end_points.reshape((1, -1, 2)) - this_end_points.reshape(
            (-1, 1, 2)
        )
        distances = np.linalg.norm(diff, axis=-1)

        min_idx = distances.argmin(axis=1)
        min_val = distances[np.arange(distances.shape[0]), min_idx]

        mask = min_val < threshold
        if min_idx[0] == min_idx[1]:
            eq_mask = np.arange(2) == np.argmax(min_val)
            mask = mask & eq_mask

        coords = np.array(line.coords)
        idx = np.array([0, -1])
        coords[idx[mask]] = other_end_points[min_idx[mask]]

        lines[i] = LineString(coords)

    buffers = [line.buffer(buffer_width) for line in lines]

    # filter lines by redundancy and length
    filtered_lines = []
    for i, line in enumerate(lines):
        other_buffers = buffers[:i] + buffers[i + 1 :]
        # check if line is fully contained in any other bbox
        is_redundant = any(b.contains(line) for b in other_buffers)
        if not is_redundant:
            filtered_lines.append(line)

    if len(filtered_lines) == 1:
        new_geom = filtered_lines[0]  # single LineString
    else:
        new_geom = MultiLineString(filtered_lines)
        new_geom = linemerge(new_geom)

    return new_geom


def snap_branches_to_main(gdf, threshold):
    new_gdf = gdf.copy()

    for i, row in new_gdf.iterrows():
        other_geoms = new_gdf.drop(i).geometry
        other_coords = []
        for geom in other_geoms:
            if isinstance(geom, MultiLineString):
                coords = [c for g in geom.geoms for c in g.coords]
            else:
                coords = geom.coords
            other_coords.extend(coords)
        other_coords = np.array(other_coords)
        
        this_geom = row.geometry
        this_lines = this_geom.geoms if isinstance(this_geom, MultiLineString) else [this_geom]
        
        new_coords = []
        for line in this_lines:
            coords = np.array(line.coords)
            end_points = coords[[0, -1]]

            # get distance
            end_points_shape, other_shape = (-1, 2, 2), (-1, 1, 2)
            diff = end_points.reshape(end_points_shape) - other_coords.reshape(other_shape)
            distances = np.linalg.norm(diff, axis=-1)
        
            min_idx = distances.argmin(axis=0)
            end_point_idx = np.arange(distances.shape[-1])
            
            min_dist = distances[min_idx, end_point_idx]
            mask = min_dist < threshold

            coords[-end_point_idx[mask]] = other_coords[min_idx[mask]]
            new_coords.append(coords)
            
        new_row = row.copy()
        new_lines = [LineString(c) for c in new_coords]
        if len(new_lines) == 1:
            new_geom = new_lines[0]
        else:
            new_geom = MultiLineString(new_lines)
        new_row["geometry"] = new_geom
        new_gdf.loc[i] = new_row

    return new_gdf

In [ ]:
def recursive_merge_lines(
    geom: LineString | MultiLineString,
) -> LineString | MultiLineString:
    if isinstance(geom, LineString):
        return geom

    # these showed good results with trial and error
    threshold = 20_000
    buffer_width = 5_000

    new_geom = merge_lines(geom, threshold=threshold, buffer_width=buffer_width)
    while not (isinstance(new_geom, LineString) or (new_geom == geom)):
        geom = new_geom
        new_geom = merge_lines(geom, threshold=threshold, buffer_width=buffer_width)
    return new_geom


original_crs = clean_rivers.crs
assert original_crs is not None

rivers_metric = clean_rivers.to_crs("EPSG:3857")
rivers_metric.geometry = rivers_metric.geometry.apply(
    recursive_merge_lines  # type: ignore
)
rivers_metric = snap_branches_to_main(rivers_metric, threshold=100_000)
# Reproject back to original CRS
rivers_final = rivers_metric.to_crs(original_crs)

rivers_final.to_file(DATA_PATH / "osm_10m_rivers_germany.geojson", driver="GeoJSON")

In [ ]:
rivers_final.plot()

In [ ]:
rivers_final.destination

In [ ]:
min_dist

In [ ]:
coords[-end_point_min[mask]] = min_dist[mask]

In [ ]:


idx = np.array([0, -1])
coords[idx[mask]] = other_end_points[min_idx[mask]]

In [ ]:
other_min, line_min, end_point_min

In [ ]:
distances[other_min, line_min, end_point_min]

In [ ]:
min_idx_flat = distances.reshape((-1, 2)).argmin(axis=0)
other_min, line_min = np.unravel_index(min_idx_flat, distances.shape[:-1])
other_min, line_min

In [ ]:
idx = e.reshape(-1, 2).argmin(axis=0)
np.unravel_index(idx, e.shape[:2])

In [ ]:
e